# MindStream — CV Emotion Model (Model 1, Improved)
TensorFlow/Keras, transfer learning (MobileNetV2/MobileNetV3/EfficientNet), FER2013 -> 7-class emotion classifier.

**Improvements over the baseline in this pass:**
- Pluggable pretrained backbone (MobileNetV2 / MobileNetV3-Large / EfficientNetB0 / EfficientNetB3 / ResNet50) instead of a from-scratch CNN.
- Stronger augmentation: flips, rotation, zoom, translation, brightness, contrast, and true random crops (resize-then-crop).
- Class imbalance handled with a **sparse focal loss** (class-weighted alpha + focal gamma) instead of plain cross-entropy — this specifically helps rare classes like `disgust`.
- Label smoothing built into the loss.
- Higher input resolution (160x160 by default, configurable up to 224x224).
- LR scheduling via `ReduceLROnPlateau` in phase 1 and **cosine-decay-with-restarts** in phase 2 fine-tuning, on top of early stopping.
- Longer training budget with early stopping so epochs aren't hand-tuned.

Two-phase training: head warmup (backbone frozen) -> fine-tune (top layers unfrozen).

In [ ]:
import os
import glob
import math
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

## Config
Paths match the `E:\\Core-AI\\projects\\mindstream` layout — edit here if yours differs.

`BACKBONE` selects the pretrained network. Bigger backbones / resolutions are more accurate but slower —
`mobilenet_v3_large` at 160x160 is a good speed/accuracy default; bump to `efficientnet_b0` (or B3) at 224x224
if you have GPU headroom.

In [ ]:
DATA_DIR = r"E:\Core-AI\DATASETS\FER2013"
CHECKPOINT_DIR = r"E:\Core-AI\MODELS\CV\checkpoints"

FER_CLASSES = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
NUM_CLASSES = len(FER_CLASSES)

# --- Backbone / resolution ---------------------------------------------------
# One of: "mobilenet_v2", "mobilenet_v3_large", "mobilenet_v3_small",
#         "efficientnet_b0", "efficientnet_b3", "resnet50"
BACKBONE = "mobilenet_v3_large"
IMG_SIZE = (160, 160)          # bump to (224, 224) for efficientnet_b3 / resnet50 if you have the VRAM/time
CROP_PAD_FRACTION = 0.15       # resize to IMG_SIZE*(1+pad) then random-crop back to IMG_SIZE (true random crops)

BATCH_SIZE = 32

# --- Training budget -----------------------------------------------------
EPOCHS_WARMUP = 15
EPOCHS_FINETUNE = 40            # early stopping will almost certainly cut this short
FINE_TUNE_UNFREEZE_FRACTION = 0.55   # unfreeze the top ~55% of backbone layers in phase 2

# --- Loss ------------------------------------------------------------------
USE_FOCAL_LOSS = True           # focal loss (gamma>0) helps rare classes like `disgust`; set False for plain CE
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.1

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RESIZE_TARGET = (
    int(round(IMG_SIZE[0] * (1 + CROP_PAD_FRACTION))),
    int(round(IMG_SIZE[1] * (1 + CROP_PAD_FRACTION))),
)
print(f"Backbone: {BACKBONE} | IMG_SIZE: {IMG_SIZE} | resize-before-crop: {RESIZE_TARGET}")

## 1. Backbone factory
Wraps the supported pretrained networks + their matching `preprocess_input` so the rest of the notebook
doesn't care which one is selected.

In [ ]:
def get_backbone(name, input_shape):
    """Returns (base_model, preprocess_input_fn) for a given backbone name."""
    name = name.lower()

    if name == "mobilenet_v2":
        from tensorflow.keras.applications import MobileNetV2
        from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
        base = MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")

    elif name == "mobilenet_v3_large":
        from tensorflow.keras.applications import MobileNetV3Large
        from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
        # include_preprocessing=False so we control preprocessing explicitly in the data pipeline
        base = MobileNetV3Large(input_shape=input_shape, include_top=False, weights="imagenet",
                                 include_preprocessing=False)

    elif name == "mobilenet_v3_small":
        from tensorflow.keras.applications import MobileNetV3Small
        from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
        base = MobileNetV3Small(input_shape=input_shape, include_top=False, weights="imagenet",
                                 include_preprocessing=False)

    elif name == "efficientnet_b0":
        from tensorflow.keras.applications import EfficientNetB0
        from tensorflow.keras.applications.efficientnet import preprocess_input
        base = EfficientNetB0(input_shape=input_shape, include_top=False, weights="imagenet")

    elif name == "efficientnet_b3":
        from tensorflow.keras.applications import EfficientNetB3
        from tensorflow.keras.applications.efficientnet import preprocess_input
        base = EfficientNetB3(input_shape=input_shape, include_top=False, weights="imagenet")

    elif name == "resnet50":
        from tensorflow.keras.applications import ResNet50
        from tensorflow.keras.applications.resnet50 import preprocess_input
        base = ResNet50(input_shape=input_shape, include_top=False, weights="imagenet")

    else:
        raise ValueError(f"Unknown BACKBONE: {name}")

    base.trainable = False
    return base, preprocess_input


# Instantiate once so both the data pipeline (preprocess_input) and the model builder
# (base architecture) agree on the exact same backbone.
_backbone_probe, preprocess_input = get_backbone(BACKBONE, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
del _backbone_probe  # we'll build the real one fresh in the model-building cell

## 2. Data augmentation + pipeline
FER2013 images are 48x48 grayscale. We resize to `RESIZE_TARGET` (slightly larger than `IMG_SIZE`) and
replicate the single channel to 3 so the pretrained ImageNet weights apply. For training we then apply a
stronger augmentation stack — including a true random crop back down to `IMG_SIZE` — and finish with the
backbone's own `preprocess_input`. Validation just center-resizes straight to `IMG_SIZE` (no crop, no aug).

In [ ]:
augmentation_layer = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.12),
        layers.RandomZoom(0.12),
        layers.RandomTranslation(0.08, 0.08),
        layers.RandomContrast(0.15),
        layers.RandomBrightness(0.15),
        layers.RandomCrop(IMG_SIZE[0], IMG_SIZE[1]),  # true random crop, resize-then-crop
    ],
    name="data_augmentation",
)

def build_dataset(directory, training=True):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="int",
        class_names=FER_CLASSES,
        color_mode="grayscale",
        image_size=(48, 48),
        batch_size=BATCH_SIZE,
        shuffle=training,
    )

    def _prep(img, label):
        img = tf.image.grayscale_to_rgb(img)
        if training:
            img = tf.image.resize(img, RESIZE_TARGET)
            img = augmentation_layer(img, training=True)
        else:
            img = tf.image.resize(img, IMG_SIZE)
        img = preprocess_input(img)
        return img, label

    return ds.map(_prep, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

### Sanity check
Confirms the pipeline loads and shapes are correct before building the model.

In [ ]:
train_dir = os.path.join(DATA_DIR, "train")
val_dir = os.path.join(DATA_DIR, "val")

train_ds = build_dataset(train_dir, training=True)
val_ds = build_dataset(val_dir, training=False)

for imgs, labels in train_ds.take(1):
    print("Batch image shape:", imgs.shape)
    print("Batch label shape:", labels.shape)
    print("Pixel value range:", float(imgs.numpy().min()), "to", float(imgs.numpy().max()))
    print("Sample labels:", labels.numpy()[:8])

## 3. Class weights
FER2013 is heavily imbalanced (e.g. `disgust` has far fewer samples than `happy`). We compute balanced
class weights here and feed them into the focal loss's `alpha` term below (rather than double-applying them
via `model.fit(class_weight=...)` as well — that would over-correct).

In [ ]:
def get_class_weights(train_dir):
    y_train = []
    for class_idx, class_name in enumerate(FER_CLASSES):
        folder_path = os.path.join(train_dir, class_name)
        if os.path.exists(folder_path):
            count = len(glob.glob(os.path.join(folder_path, "*.*")))
            y_train.extend([class_idx] * count)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=np.array(y_train)
    )
    return dict(enumerate(weights))

class_weights = get_class_weights(train_dir)
print("Class weights:", class_weights)

# Normalize into an alpha vector for the focal loss (mean ~1 so overall loss scale stays sane)
_alpha_raw = np.array([class_weights[i] for i in range(NUM_CLASSES)], dtype=np.float32)
alpha_vector = _alpha_raw / _alpha_raw.mean()
print("Focal loss alpha vector:", dict(zip(FER_CLASSES, alpha_vector.round(3))))

## 4. Loss: sparse focal loss + label smoothing
Standard `SparseCategoricalCrossentropy` in TF/Keras doesn't support label smoothing, and plain
cross-entropy under-weights hard/rare examples like `disgust`. This custom loss:
- one-hot encodes the sparse integer labels internally,
- applies label smoothing,
- applies the focal modulating factor `(1 - p_t)^gamma` (set `FOCAL_GAMMA=0` to disable and fall back to
  plain smoothed cross-entropy),
- applies the per-class `alpha` weighting computed from the balanced class weights above.

In [ ]:
class SparseFocalLossWithSmoothing(tf.keras.losses.Loss):
    def __init__(self, num_classes, gamma=2.0, alpha=None, label_smoothing=0.0,
                 name="sparse_focal_loss_with_smoothing"):
        super().__init__(name=name)
        self.num_classes = num_classes
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.alpha = None if alpha is None else tf.constant(alpha, dtype=tf.float32)

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_true_onehot = tf.one_hot(y_true, depth=self.num_classes)

        if self.label_smoothing > 0:
            y_true_smooth = y_true_onehot * (1.0 - self.label_smoothing) + \
                             self.label_smoothing / self.num_classes
        else:
            y_true_smooth = y_true_onehot

        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        ce = -y_true_smooth * tf.math.log(y_pred)
        focal_weight = tf.pow(1.0 - y_pred, self.gamma) if self.gamma > 0 else 1.0
        loss = focal_weight * ce

        if self.alpha is not None:
            # per-example alpha = alpha of its (hard) true class
            alpha_t = tf.gather(self.alpha, y_true)
            loss = loss * alpha_t[:, tf.newaxis]

        return tf.reduce_sum(loss, axis=-1)


loss_fn = SparseFocalLossWithSmoothing(
    num_classes=NUM_CLASSES,
    gamma=FOCAL_GAMMA if USE_FOCAL_LOSS else 0.0,
    alpha=alpha_vector,
    label_smoothing=LABEL_SMOOTHING,
)
print(loss_fn)

## 5. Model architecture
Pretrained backbone (frozen initially, selected via `BACKBONE`) + a wider classifier head with BatchNorm
and dropout.

In [ ]:
def build_model(backbone_name, input_shape, num_classes):
    base_model, _ = get_backbone(backbone_name, input_shape)  # base_model.trainable = False already

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = models.Model(inputs, outputs, name=f"MindStream_{backbone_name}")
    return model, base_model

model, base_model = build_model(BACKBONE, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=NUM_CLASSES)
model.summary()

## 6. Callbacks
Best-checkpoint saving, LR reduction on plateau, early stopping (so epochs aren't hand-tuned), and a CSV
log for later inspection.

In [ ]:
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_emotion_model.keras")
log_path = os.path.join(CHECKPOINT_DIR, "training_log.csv")

cb_list = [
    callbacks.ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
    callbacks.CSVLogger(log_path, append=True),
]

## Phase 1 — Train classification head (backbone frozen)

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss=loss_fn,
    metrics=["accuracy"]
)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_WARMUP,
    callbacks=cb_list
)

## Phase 2 — Fine-tune the backbone
Unfreezes the top `FINE_TUNE_UNFREEZE_FRACTION` of the backbone's layers. Uses a **cosine-decay-with-restarts**
schedule (instead of a single tiny fixed LR) so the optimizer keeps exploring across the extended training
budget, with `ReduceLROnPlateau`/`EarlyStopping` from `cb_list` still active as guardrails.

In [ ]:
base_model.trainable = True

n_layers = len(base_model.layers)
fine_tune_at = int(n_layers * (1 - FINE_TUNE_UNFREEZE_FRACTION))
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
print(f"Unfreezing layers {fine_tune_at}..{n_layers} of {n_layers} in {BACKBONE}")

steps_per_epoch = tf.data.experimental.cardinality(train_ds).numpy()
if steps_per_epoch < 0:  # unknown cardinality fallback
    steps_per_epoch = sum(1 for _ in train_ds)

lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-4,
    first_decay_steps=steps_per_epoch * 5,   # restart roughly every 5 epochs
    t_mul=1.5,
    m_mul=0.85,
    alpha=1e-2,  # floor as a fraction of initial_learning_rate
)

model.compile(
    optimizer=optimizers.Adam(learning_rate=lr_schedule),
    loss=loss_fn,
    metrics=["accuracy"]
)

# Note: with a schedule-driven LR, ReduceLROnPlateau in cb_list is a no-op on the optimizer's LR tensor;
# EarlyStopping/ModelCheckpoint/CSVLogger still function normally.
fine_tune_cb_list = [cb for cb in cb_list if not isinstance(cb, callbacks.ReduceLROnPlateau)]

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_WARMUP + EPOCHS_FINETUNE,
    initial_epoch=history_phase1.epoch[-1] + 1,
    callbacks=fine_tune_cb_list
)

print(f"\nTraining complete. Best checkpoint saved to:\n  {checkpoint_path}")

## 7. Evaluation
Classification report + confusion matrix + macro-F1 on the held-out val set, using the best saved
checkpoint. Macro-F1 (unweighted across classes) is a better health check than accuracy alone for an
imbalanced dataset like FER2013.

In [ ]:
best_model = tf.keras.models.load_model(
    checkpoint_path,
    custom_objects={"SparseFocalLossWithSmoothing": SparseFocalLossWithSmoothing},
)

y_true, y_pred = [], []
for imgs, labels in val_ds:
    preds = best_model.predict(imgs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=FER_CLASSES))
print(confusion_matrix(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))